In [12]:
import pandas as pd
import re
import os
import sys
from pathlib import Path

# we go up two levels to find the root
current_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
project_root = current_dir.parents[1] # This gets us to 'sentiment-analysis-tool'

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root)) # Put it at the very top of the list

# IMPORT CUSTOM UTILS
try:
    from src.utils.config import get_path
    print("System paths connected successfully.")
except ImportError as e:
    print(f"Can't find the config file: {e}")
    print(f"Looking in: {project_root / 'src' / 'utils'}")
    sys.exit(1)

# SET UP DIRECTORIES
raw_folder = get_path("data", "raw_data")
output_folder = get_path("data", "anonymized")
output_folder.mkdir(parents=True, exist_ok=True)

# PROCESS EXCEL FILES
excel_files = list(raw_folder.glob("*.xlsx"))

if not excel_files:
    print(f"No files found in {raw_folder}. Check your folder path!")
    sys.exit()

df_list = []
for file_path in excel_files:
    try:
        # engine='openpyxl' is required for Python 3.11 + .xlsx
        df = pd.read_excel(file_path, engine='openpyxl')
        df_list.append(df)
    except Exception as e:
        print(f"Skipping {file_path.name}: {e}")

if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # ANONYMIZE
    def anonymize_text(text):
        if not isinstance(text, str) or pd.isnull(text):
            return text
        text = re.sub(r'\S+@\S+', '[EMAIL]', text)
        text = re.sub(r'\b(?:\+254|0)?7\d{8}\b', '[PHONE]', text)
        text = re.sub(r'\b\d{8}\b', '[ID_NUMBER]', text)
        text = re.sub(r'\b\d{10,}\b', '[ACCOUNT_NUMBER]', text)
        text = re.sub(r'\b[A-Z][a-z]+\b', '[NAME]', text)
        return text

    if "text" in combined_df.columns:
        combined_df["text"] = combined_df["text"].apply(anonymize_text)
        
    # SAVE
    output_file = output_folder / "customer_feedback_anonymized.xlsx"
    combined_df.to_excel(output_file, index=False, engine='openpyxl')
    print(f"Done! Combined {len(excel_files)} files into {output_file}")

Can't find the config file: cannot import name 'get_path' from 'src.utils.config' (c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\src\utils\config.py)
Looking in: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\src\utils


SystemExit: 1